# Travel Itinerary Assistant: full run


In [1]:
import travel_agent.config  # loads .env and sets the API key
from agents import Runner
from travel_agent.agents.intake_agent import intake_agent
from travel_agent.pipeline import build_plan

## 1. Intake: multi-turn chat


In [2]:
history = []

async def say(msg):
    global history
    result = await Runner.run(intake_agent, history + [{"role": "user", "content": msg}])
    history = result.to_input_list()
    out = result.final_output
    print("missing:", out.missing_fields)
    print("assistant:", out.next_question)
    return out

out = await say("I want to visit Japan with 2 friends starting October 26 for 7 days, budget 6000 USD")

missing: ['passport_country', 'origin', 'interests']
assistant: What country issued your passport?


In [3]:
out = await say("I hold a French passport, flying from Miami. Food and temples for vacation.")

missing: []
assistant: None


In [4]:
assert out.is_complete, out.missing_fields
trip = out.trip
print(trip.model_dump_json(indent=2))

{
  "trip_type": "leisure",
  "passport_country": "France",
  "origin": "Miami",
  "destination": "Japan",
  "start_date": "2026-10-26",
  "end_date": "2026-11-01",
  "num_days": 7,
  "travelers": 3,
  "budget_usd": 6000.0,
  "interests": [
    "food",
    "temples"
  ],
  "meeting_address": null,
  "needs_car_rental": null
}


## 2. Tool smoke test (no LLM)

Confirms mock data files and tools work before running agents.


In [5]:
from travel_agent.tools.transport_tool import search_flights_raw
from travel_agent.tools.stay_tool import search_stays_raw
from travel_agent.tools.visa_tool import check_visa_raw
from travel_agent.airports import resolve_airport

origin, dest = resolve_airport(trip.origin), resolve_airport(trip.destination)
print(origin, dest)
print((await search_flights_raw(origin, dest, trip.start_date, trip.end_date, trip.travelers, force_mock=True))["source"])
print(search_stays_raw(trip.destination, trip.num_days - 1, trip.travelers)["options"][:1])
print(check_visa_raw(trip.passport_country, trip.destination)["source"])

MIA NRT
mock
[{'name': 'Sample Tokyo City Hotel', 'area': 'Tokyo', 'total_cost_usd': 1500, 'why': 'Sample central option with convenient access to restaurants, transit, and major attractions.', 'is_estimate': True}]
mock


## 3. Full pipeline

Visa, Activities, Transport and Stay run in parallel. Then budget, then Planner.


In [6]:
plan = await build_plan(trip)
print("warnings:", plan.warnings)
print(plan.budget.model_dump_json(indent=2))

warnings: []
{
  "flights_usd": 2400.0,
  "stay_usd": 1020.0,
  "activities_usd": 15.0,
  "local_transport_usd": 0.0,
  "food_usd": 945.0,
  "total_usd": 4380.0,
  "budget_usd": 6000.0,
  "within_budget": true,
  "remaining_usd": 1620.0
}


In [7]:
it = plan.itinerary
print(it.title)
print(it.summary)
print("Visa:", it.visa_note)
print("Budget:", it.budget_note)
for d in it.days:
    print(f"Day {d.day} | {d.date} | {d.city}")
    for item in d.items:
        print(" -", item)

7-Day Kyoto Leisure Trip Itinerary
This 7-day itinerary in Kyoto showcases a delightful mix of cultural temple visits and food experiences that align with your interests. Each day features 2 to 4 distinct activities with ample free time to relax and explore independently. There are no day trips outside Kyoto as per your preference, and each activity is unique without repetition. Since activities are fewer than might fill every moment, planned free time ensures a comfortable pace. Day 1 includes arrival with airport transfer from Miami (MIA) to Narita (NRT), which is not priced. Day 7 includes your departure back to Miami, with airport transfer not priced as well.
Visa: French passport holders can enter Japan without a visa for up to 90 days for tourism or business purposes. This information is from a mock source; please verify visa requirements on the official Japanese government website before travel.
Budget: Total estimated cost for flights, stay, activities, and food is $4380.00 USD